In [ ]:
import pandas as pd 
import numpy as np 
from sklearn.linear_model import LogisticRegression
from feats import elo

df = pd.read_csv("history.csv")

def h2h(df: pd.DataFrame, date: pd.DatetimeIndex, HomeTeam: str, AwayTeam: str, games = 5):
    # limit dates to only the games before the match, excluding it
    df = df[df["Date"] < date]
    # choose games where the teams face each other
    df = df[(df["HomeTeam"] == HomeTeam) | (df["HomeTeam"] == AwayTeam)]
    df = df[(df["AwayTeam"] == HomeTeam) | (df["AwayTeam"] == AwayTeam)]
    df = df.tail(games)
    score = [0,0,0]
    for row in df.itertuples():
        if row.FTR == "H":
            score[0] += 1
        elif row.FTR == "D":
            score [1] += 1
        else: 
            score[2] += 1
    return score 

In [ ]:
# Run to generate new WR and MP -> change the value

df = pd.read_csv("history.csv", low_memory=True)
if "WR" in df.columns:
    df.drop(columns=["WR","MP"], inplace=True)

wrMargin = []
matchesPlayed = []
for match in df.itertuples():
    record = h2h(df, match.Date, match.HomeTeam, match.AwayTeam, 10)
    wrMargin.append((record[0] - record[2])/ (record[0] + record[1] + record[2] + 1**-5))
    matchesPlayed.append(record[0] + record[1] + record[2])
data = {"WR": wrMargin, "MP": matchesPlayed}

dfRecords = pd.DataFrame(data)

feats = pd.concat([df,dfRecords], axis=1)
feats
feats.to_csv("history.csv", index=False)

In [ ]:

df = pd.read_csv("history.csv")
featDf = pd.DataFrame(elo(df))
data = pd.concat([df, featDf], axis=1)
data["elo_diff"] = (data["home_elo_pre_match"] - data["away_elo_pre_match"]) 
# feats = pd.concat([data,dfRecords],axis=1)
feats = data

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, accuracy_score
model = LogisticRegression(max_iter=1000)
le = LabelEncoder()
categories = ["H", "D", "A"]

le.fit(categories)

train = feats[(feats["season"] > 2011) & (feats["season"] < 2023)]
test = feats[feats["season"] >= 2023]

trainX = train[["WR","MP","elo_diff"]]
trainY = train["FTR"]

testX = test[["WR","MP","elo_diff"]]
testY = test["FTR"]

model = model.fit(trainX,trainY)

probs = model.predict_proba(testX)
preds = model.predict(testX)
 
model_logloss = log_loss(testY, probs, labels=model.classes_)
model_acc = accuracy_score(testY, preds)
 
naive_probs = np.tile([1/3, 1/3, 1/3], (len(testY), 1))  # coin-flip 3-way baseline
naive_logloss = log_loss(testY, naive_probs, labels=model.classes_)
 
print(f"\nLogistic regression -- log loss: {model_logloss:.4f}, accuracy: {model_acc:.3f}")
print(f"Naive uniform baseline -- log loss: {naive_logloss:.4f}")

In [ ]:
# TODO: test with elo and compare log loss, use walk forward validation